In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
#import ipywidgets as widgets
from IPython.display import display, Video, clear_output

import cv2
import mediapipe as mp
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

Чтение

In [8]:
data = pd.read_csv('data/annotations.csv', sep='\t', on_bad_lines='skip')
data

,attachment_id,text,user_id,height,width,length,train,begin,end
0,44e8d2a0-7e01-450b-90b0-beb7400d2c1e,Ё,185bd3a81d9d618518d10abebf0d17a8,1920,1080,156.0,True,36,112
1,df5b08f0-41d1-4572-889c-8b893e71069b,А,185bd3a81d9d618518d10abebf0d17a8,1920,1080,150.0,True,36,76
2,17f53df4-c467-4aff-9f48-20687b63d49a,Р,185bd3a81d9d618518d10abebf0d17a8,1920,1080,133.0,True,40,97
3,e3add916-c708-4339-ad98-7e2740be29e9,Е,185bd3a81d9d618518d10abebf0d17a8,1920,1080,144.0,True,43,107
4,bd7272ed-1850-48f1-a2a8-c8fed523dc37,Ч,185bd3a81d9d618518d10abebf0d17a8,1920,1080,96.0,True,20,70
...,...,...,...,...,...,...,...,...,...
20395,nodca88242-2bc7-4a77-9d14-103aa1dacbd6,no_event,0041ec866777f12c384b64d8cd636277,1920,1080,42.0,False,0,42
20396,no7a7812b1-ae64-4402-9ebc-5b947edbd021,no_event,f5b82a9c82f6d870ec253e4c3fa96d83,1280,720,32.0,False,0,32
20397,no62a4df76-b48d-4f61-b6e8-bc4a1eb0cb61,no_event,4299b8ccf39ace57287b463fbe4a489b,1920,960,32.0,False,0,32
20398,no388a3f7c-3594-4332-bc78-b5b53190301d,no_event,c80b4e57f158f28299b2a89694c42329,1920,1080,41.0,False,0,41


In [3]:
data = data.head(1000)

In [9]:
source_directory = 'data' # базовая папка
train_dir = os.path.join(source_directory, 'dataset_train_proc')
test_dir = os.path.join(source_directory, 'dataset_test_proc')

In [10]:
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

Label2ind classes:

In [11]:
all_class_counts = data['text'].value_counts()
top_classes = all_class_counts.index.tolist()
data = data[data['text'].isin(top_classes)]

# Словарь меток только для этих классов
class_names = sorted(top_classes)
label2idx = {label: idx for idx, label in enumerate(class_names)}

In [12]:
len(class_names)

1001

Параметры данных:

In [13]:
# Параметры

SEQUENCE_LENGTH = 48
MAX_HANDS = 2
NUM_LANDMARKS = 21 #количество точек для каждой руки
NUM_FEATURES_PER_LANDMARK_HAND = 3 #количество координат для каждой точки
NUM_FEATURES = MAX_HANDS * NUM_LANDMARKS * NUM_FEATURES_PER_LANDMARK_HAND  #21 точка по 3 координаты

NUM_FEATURES_PER_LANDMARK = 2
LIPS_IDX = [61, 37, 0, 267, 291, 405, 17, 181] #точки губ
NUM_FEATURES_FACE = len(LIPS_IDX)

POSE_IDX = [16, 14, 12, 11, 13, 15, 0] #точки рук + нос
NUM_FEATURES_POSE = len(POSE_IDX)

NUM_FEATURES_ALL = NUM_FEATURES + NUM_FEATURES_FACE * NUM_FEATURES_PER_LANDMARK + NUM_FEATURES_POSE * NUM_FEATURES_PER_LANDMARK


Нормализация данных:

In [14]:
def normalize_and_scale(landmarks):

    landmarks = np.array(landmarks)
    #print(landmarks.shape)
    check = True
    data_is_arr = []
    arr_hands = np.array(landmarks[:, :NUM_FEATURES]).reshape(landmarks.shape[0], -1, 3)
    arr_lips_pose = np.array(landmarks[:, NUM_FEATURES:]).reshape(landmarks.shape[0], -1, 2)

    for l in landmarks:
      data = [0, 0, 0, 0] #np.any(np.array(data_is_arr[:2]) != 0)
      if np.any(l[:NUM_LANDMARKS * NUM_FEATURES_PER_LANDMARK_HAND] != 0):
        data[0] = 1
      if np.any(l[NUM_LANDMARKS * NUM_FEATURES_PER_LANDMARK_HAND:NUM_FEATURES] != 0):
        data[1] = 1
      if np.any(l[NUM_FEATURES:NUM_FEATURES + NUM_FEATURES_FACE * NUM_FEATURES_PER_LANDMARK] != 0):
        data[2] = 1
      if np.any(l[NUM_FEATURES + NUM_FEATURES_FACE * NUM_FEATURES_PER_LANDMARK:] != 0):
        data[3] = 1
      data_is_arr.append(data)
      
    res = [sum(col) for col in zip(*data_is_arr)]

    if res[-1] > SEQUENCE_LENGTH * 0.1: # pose
      left_border_idx = NUM_FEATURES_FACE + 3  # Левое плечо
      right_border_idx = NUM_FEATURES_FACE + 2 # Правое плечо

    elif res[-2] > SEQUENCE_LENGTH * 0.1: #lips
      left_border_idx = 4  # Левый уголок рта
      right_border_idx = 0 # Правый уголок рта

    elif res[0] > 0 or res[1] > 0: # only hands
      check = False
      wrist = arr_hands[:, 0, :]

      # Центрирование относительно запястья
      arr_hands[:, :, :2] -= wrist

      # Масштабирование относительно длины руки
      index_finger_base = arr_hands[:, 5, :]  # Основание указательного пальца
      middle_finger_tip = arr_hands[:, 12, :] # Кончик среднего пальца
      scale = np.linalg.norm(middle_finger_tip - index_finger_base)

      if scale > 0.01:  # Защита от деления на ноль
          arr_hands[:, :, :2] /= scale

      arr_lips_pose = np.full_like(arr_lips_pose, np.nan, dtype=float)

    else:
      check = False

    if check:
      # Центрирование относительно центра
      center = (arr_lips_pose[:, left_border_idx, :] + arr_lips_pose[:, right_border_idx, :]) / 2
      center = center[:, np.newaxis, :]


      mask = np.isnan(center).all(axis=2)  # True для NaN
      for t in range(1, center.shape[0]):
          center[t][mask[t]] = center[t-1][mask[t]]

      arr_lips_pose -= center
      arr_hands[:, :, :2] -= center

      arr_hands[:, :, 2:] *= 0.5

      # Масштабирование относительно длины руки

      scale = np.linalg.norm(arr_lips_pose[:, left_border_idx] - arr_lips_pose[:, right_border_idx])

      if scale > 0.001:  # Защита от деления на ноль
          arr_lips_pose /= scale
          arr_hands[:, :2] /= scale

    arr = np.concatenate(
    [
        arr_hands.reshape(arr_hands.shape[0], -1),
        arr_lips_pose.reshape(arr_lips_pose.shape[0], -1)
    ],
    axis=1)

    return arr#.flatten()

Чтение + нормализация файлов:

In [15]:
def find_npy_path(attachment_id, is_train=True):
  folder = train_dir if is_train else test_dir
  path = os.path.join(folder, f"{attachment_id}.npy")
  if os.path.exists(path):
    return path
  return None

In [16]:
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []



for idx, row in tqdm(data.iterrows(), total=len(data)):
    attachment_id = row['attachment_id']

    # Пробуем train
    video_path = find_npy_path(attachment_id, is_train=True)
    if video_path:
        seq = np.load(video_path)
        X_train_list.append(normalize_and_scale(seq))
        y_train_list.append(label2idx[row['text']])
        continue

    # Пробуем test
    video_path = find_npy_path(attachment_id, is_train=False)
    if video_path:
        seq = np.load(video_path)
        X_test_list.append(seq)
        y_test_list.append(label2idx[row['text']])
        continue

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)
X_test = np.array(X_test_list)
y_test = np.array(y_test_list)

100%|██████████| 20400/20400 [07:23<00:00, 45.97it/s]


In [17]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


In [33]:
class GestureDataset(Dataset):
    def __init__(self, data, transform=None):
        """
        data_dir: путь к папке, где лежат подпапки с классами
        class_names: список названий классов, например ['no-gesture', 'да', 'нет', ...]
        """
        self.data = []
        self.labels = []
        self.transform = transform

        for idx, row in tqdm(data.iterrows(), total=len(data)):
            attachment_id = row['attachment_id']
            
            video_path = find_npy_path(attachment_id, is_train=True)
            if video_path:
                seq = np.load(video_path)
                self.data.append(normalize_and_scale(seq))
                self.labels.append(label2idx[row['text']])

        print(f"Загружено {len(self.data)} примеров")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        label = self.labels[idx]

        if self.transform:
            seq = self.transform(seq)

        return torch.from_numpy(seq).float(), torch.tensor(label, dtype=torch.long)


# Пример использования
data_dir = train_dir  # замените на реальный путь
#class_names = data['text'].unique()  # или ваши реальные имена

full_dataset = GestureDataset(data=data)

# Разделение на train / val (stratified)
train_idx, val_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.15,
    #stratify=full_dataset.labels,
    random_state=42
)

train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
val_dataset   = torch.utils.data.Subset(full_dataset, val_idx)

# DataLoader'ы
train_loader = DataLoader(
    train_dataset,
    batch_size=64,#64,
    shuffle=True,
    #num_workers=1,          # зависит от вашего CPU
    #pin_memory=True,        # ускоряет перенос на GPU
    drop_last=True          # полезно для contrastive learning
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,#64,
    shuffle=False
    #num_workers=1,
    #pin_memory=True
)

100%|██████████| 20400/20400 [01:28<00:00, 231.65it/s] 

Загружено 15300 примеров


Модель:

In [18]:
import torch.nn as nn
import torch.nn.functional as F
import torch

In [19]:
import torch.optim as optim
from torch.utils.data import DataLoader

In [20]:
class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = (torch.device('cuda')
                  if features.is_cuda
                  else torch.device('cpu'))

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss


class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0, dim=-1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            # true_dist = pred.data.clone()
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))


LOSSES = {'SupCon': SupConLoss,
          'LabelSmoothing': LabelSmoothingLoss,
          'CrossEntropy': nn.CrossEntropyLoss}

In [21]:
import math

In [22]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        
    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True) * (x.shape[-1] ** -0.5)
        return x / (norm + self.eps) * self.weight

class PatchEmbedding(nn.Module):
    def __init__(self, num_landmarks, d_model, patch_size=64, num_coord = 3):
        super().__init__()
        self.patch_size = patch_size * num_coord #size патчей
        self.num_landmarks = num_landmarks #количество точек ключевых
        self.d_model = d_model
        
        self.proj = nn.Linear(patch_size * num_coord, d_model)
        self.norm = RMSNorm(d_model) #слой нормализации
        
        max_patches = (num_landmarks + patch_size - 1) // patch_size
        self.pos_embed = nn.Parameter(torch.randn(1, max_patches, d_model) * 0.02) #позиционная эмбеддинга для патчей, чтобы модель понимала порядок патчей.
        
    def forward(self, x): #first_hand
       
        B, T, N = x.shape
        
        patches = []
        for i in range(0, N, self.patch_size): # разбивка на патчи
            end_idx = min(i + self.patch_size, N)
            patch = x[:, :, i:end_idx]  #B, T, 
            
            if patch.shape[-1] < self.patch_size:
                pad_size = self.patch_size - patch.shape[-1]
                patch = F.pad(patch, (0, pad_size))
            
            patches.append(patch)
        
        patches = torch.stack(patches, dim=2)  # [B, T, num_patches, patch_size*2]
        
        patches = self.proj(patches)  # [B, T, num_patches, d_model]
        
        num_patches = patches.shape[2]
        patches = patches + self.pos_embed[:, :num_patches, :] 
        
        return self.norm(patches)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=256, num_heads=4, dropout=0.2):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        
        self.attention = nn.MultiheadAttention(self.d_model, self.num_heads)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        #B, T, D = x.shape #3072, 6, 512
        
        Q = self.w_q(x)#.view(B, T, self.d_k).transpose(1, 2)
        K = self.w_k(x)#.view(B, T, self.d_k).transpose(1, 2)
        V = self.w_v(x)#.view(B, T, self.d_k).transpose(1, 2)      

        attn_out, _ = self.attention(Q, K, V, mask)
        
        #attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        
        out = self.w_o(attn_out)
        return self.dropout(out)

In [24]:
class LandmarkEmbedding(nn.Module):
    def __init__(self, num_landmarks, d_model, patch_size=4, num_coord = 3): #LandmarkEmbedding(21, self.d_model)
        super().__init__()
        self.num_landmarks = num_landmarks
        self.d_model = d_model
        
        self.patch_embedding = PatchEmbedding(num_landmarks, d_model, patch_size, num_coord)
        
        self.self_attention = MultiHeadAttention(d_model, 4)
        
        self.missing_embed = nn.Parameter(torch.randn(d_model) * 0.02)
        
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        
    def forward(self, x): #first_hand
        # x: [B, T, N] #B, T, 63 или 16 или 14
        B, T, N = x.shape
        
        frame_missing = (x.sum(dim=(2)) == 0)  # [B, T]
        
        patches = self.patch_embedding(x)  # [B, T, num_patches, d_model]
        
        B, T, P, D = patches.shape
        patches_flat = patches.view(B * T, P, D)
        attended_patches = self.self_attention(patches_flat)
        attended_patches = attended_patches.view(B, T, P, D)
        
        temporal_repr = attended_patches.mean(dim=2)  # [B, T, d_model]
        
        missing_expanded = self.missing_embed.expand(B, T, D)
        temporal_repr = torch.where(
            frame_missing.unsqueeze(-1).expand_as(temporal_repr),
            missing_expanded,
            temporal_repr
        )
        
        return temporal_repr #temporal_repr


In [25]:
class CrossModalAttention(nn.Module):
    def __init__(self, d_model, num_heads=8):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(0.2)
        self.norm = RMSNorm(d_model)
        
    def forward(self, query, key_value):
        B, T, D = query.shape
        
        Q = self.w_q(query).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key_value).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(key_value).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, V)
        
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        
        attended = self.w_o(attn_out)
        attended = self.dropout(attended)
        
        return self.norm(query + attended)

In [ ]:
class AllLandmarkEmbedding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.first_hand_emb = LandmarkEmbedding(NUM_LANDMARKS, self.d_model)
        self.sec_hand_emb = LandmarkEmbedding(NUM_LANDMARKS, self.d_model)
        self.lips_emb = LandmarkEmbedding(NUM_FEATURES_FACE, self.d_model, num_coord=2)
        self.pose_emb = LandmarkEmbedding(NUM_FEATURES_POSE, self.d_model, num_coord=2)

        self.cross_attn_hands = CrossModalAttention(d_model)
        self.cross_attn_pose_lips = CrossModalAttention(d_model)

        #что-то для соединения
        self.fusion_weights = nn.Parameter(torch.ones(4) / 4)
        self.fusion_proj = nn.Linear(d_model * 4, d_model)
        
        self.temporal_embed = nn.Embedding(1000, d_model)
        

    def forward(self, first_hand, sec_hand, pose, lips):
        B, T = lips.shape[:2]
                       
        first_hand_emb = self.first_hand_emb(first_hand) # [B, T, d_model]
        sec_hand_emb = self.sec_hand_emb(sec_hand)
        lips_emb = self.lips_emb(lips)
        pose_emb = self.pose_emb(pose)
        
        f_hand_enhanced = self.cross_attn_hands(first_hand_emb, sec_hand_emb)
        s_hand_enhanced = self.cross_attn_hands(sec_hand_emb, first_hand_emb)
        
        lips_enhanced = self.cross_attn_pose_lips(lips_emb, pose_emb)
        pose_enhanced = self.cross_attn_pose_lips(pose_emb, lips_emb)
        
        all_features = torch.stack([
            lips_enhanced, f_hand_enhanced, s_hand_enhanced, pose_enhanced
        ], dim=-1)  # [B, T, d_model, 4]
        
        weights = F.softmax(self.fusion_weights, dim=0)
        fused = (all_features * weights).sum(dim=-1)  # [B, T, d_model]
        
        concat_features = torch.cat([
            lips_enhanced, f_hand_enhanced, s_hand_enhanced, pose_enhanced
        ], dim=-1)  # [B, T, d_model*4]
        
        projected = self.fusion_proj(concat_features)  # [B, T, d_model]
    
        x = fused + projected
        
        positions = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.temporal_embed(positions)
        
        return x



In [39]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, dropout = 0.2):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        #self.attention = MultiHeadAttention(d_model, num_heads)
        self.attention = nn.MultiheadAttention(embed_dim = d_model, num_heads = num_heads, dropout = dropout, batch_first=True)

        self.alpha_attn = nn.Parameter(torch.ones(1))

        self.ffd = nn.Sequential(nn.Linear(d_model, d_model * 4),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(d_model * 4, d_model)
                    )

        self.alpha_ffd = nn.Parameter(torch.ones(1))

    def forward(self, x, mask = None):
        attn, w = self.attention(query = x, key = x, value = x, key_padding_mask = mask, need_weights=False)
        x = x + self.alpha_attn * self.norm1(attn)

        ffd = self.ffd(x)

        x = x + self.alpha_ffd * self.norm2(ffd) 

        return x

In [28]:
class MeanMaxPooling(nn.Module):
    def __init__(self, d_model, dropout=0.2):
        super().__init__()
        
        self.proj = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            RMSNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x, attention_mask):
        """
        x: [B, T, D]
        attention_mask: [B, T] (1 — valid, 0 — padding)
        """
        # [B, T, 1]
        mask = attention_mask.unsqueeze(-1)

        # ---- Mean pooling ----
        masked_x = x * mask
        mean_pool = masked_x.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        # ---- Max pooling ----
        masked_x_max = x.masked_fill(mask == 0, -float('inf'))
        max_pool, _ = masked_x_max.max(dim=1)
        max_pool = torch.where(
            torch.isinf(max_pool),
            torch.zeros_like(max_pool),
            max_pool
        )

        # ---- Combine ----
        pooled = torch.cat([mean_pool, max_pool], dim=-1)  # [B, 2D]
        return self.proj(pooled)  # [B, D]


In [ ]:
class GestureModel(nn.Module):
    def __init__(self, d_model=256, num_heads=4, num_class = 1001):
        super().__init__()
        self.embedding = AllLandmarkEmbedding(d_model)
        self.transformer = Transformer(d_model=d_model, num_heads=num_heads)
        self.pooling = MeanMaxPooling(d_model)
        self.final_norm = RMSNorm(d_model)

        self.classificator = nn.Linear(d_model, num_class)

    def forward(self, frames):
        #attention_mask = (non_empty_frame_idxs != -1).float()
        attention_mask = (frames.abs().sum(dim=-1) != 0).int()

        x = frames[:, :, :] #[batch, seq_len, n_features]

        FIRST_HAND_START = 0
        SECOND_HAND_START = NUM_LANDMARKS * NUM_FEATURES_PER_LANDMARK_HAND 
        LIPS_START = NUM_FEATURES 
        POSE_START = NUM_FEATURES + NUM_FEATURES_FACE * NUM_FEATURES_PER_LANDMARK        
        
        first_hand = x[:, :, FIRST_HAND_START:SECOND_HAND_START] #B, T, 63
        sec_hand = x[:, :, SECOND_HAND_START:LIPS_START] #B, T, 63
        lips = x[:, :, LIPS_START:POSE_START] #B, T, 16
        pose = x[:, :, POSE_START:] #B, T, 14

        x = self.embedding(first_hand, sec_hand, pose, lips)

        x = self.transformer(x)

        x = self.pooling(x, attention_mask)

        x = self.final_norm(x)

        x = self.classificator(x)

        return x



In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GestureModel()
model.to(device)

model.train()

for sequences, labels in train_loader:
    sequences = sequences.to(device)   # [batch, seq_len, n_features]

    embeds = model(sequences)

    break

d:\Programs\anac\envs\vkr_2\Lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


In [ ]:
########
class GestureEmbedder(nn.Module):
    def __init__(self, n_features, seq_len, embed_dim=128):
        super().__init__()
        self.proj = nn.Linear(n_features, embed_dim)  # Проекция keypoints
        #self.pos_enc = nn.Parameter(torch.randn(1, seq_len, embed_dim))  # Позиционное
        #encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dim_feedforward=256)
        #self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        #self.pool = nn.AdaptiveAvgPool1d(1)  # Global average pooling
        self.head = nn.Linear(embed_dim, embed_dim)  # Для embeddings
    
    def forward(self, x):
        # x: [batch, seq_len, n_features]
        x = x.float()
        batch_size, seq_len, n_features = x.shape
        x = x.view(batch_size * seq_len, n_features)
        #x = self.proj(x) + self.pos_enc
        #x = self.transformer(x.transpose(0,1)).transpose(0,1)  # [seq_len, batch, dim] для Transformer
        #x = self.pool(x.transpose(1,2)).squeeze(-1)  # [batch, embed_dim]
        x = self.proj(x)
        x = x.view(batch_size, seq_len, -1)
        return F.normalize(self.head(x), dim=-1)  # L2-norm embeddings

In [60]:
def augment(x, sigma=0.01, shift_prob=0.5, scale_prob=0.6):
    noise = torch.randn_like(x) * sigma
    x = x + noise

    if torch.rand(1).item() < scale_prob:
        scale = torch.FloatTensor(1).uniform_(0.92, 1.08).to(x.device)
        x = x * scale

    if torch.rand(1).item() < shift_prob:
        shift = torch.randn_like(x) * 0.03
        x = x + shift


    return x

In [42]:
def focal_loss_with_smoothing(logits, targets, label_smoothing = 0.1, focal_gamma = 1, focal_alpha = 2):
        
    num_classes = logits.shape[1]
    # num_classes = NUM_CLASSES
    targets_one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1)
    targets_smooth = targets_one_hot * (1 - label_smoothing) + label_smoothing / num_classes
    
    log_probs = F.log_softmax(logits, dim=1)
    probs = torch.exp(log_probs)
    
    focal_weights = focal_alpha * (1 - probs) ** focal_gamma
    focal_loss = -(focal_weights * targets_smooth * log_probs).sum(dim=1)
    
    return focal_loss.mean()

In [ ]:

# from your_dataset import GestureDataset  # ваш датасет

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Модель
model = GestureModel()
model.to(device)

# Гиперпараметры
num_epochs = 30          # для начала, можно 100–300
print_every = 1

# Loss и оптимизатор
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)   # lr можно 1e-4 – 5e-4
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=5e-4, 
    epochs=num_epochs, 
    steps_per_epoch=len(train_loader),
    pct_start=0.15,      # 10% эпох на warmup
    anneal_strategy='cos',
    div_factor=25,                  # начальный lr = max_lr / 25 ≈ 2e-5
    final_div_factor=1000
)


# ------------------- Training loop -------------------
for epoch in tqdm(range(num_epochs)):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for sequences, labels in train_loader:
        sequences = sequences.to(device)   # [batch, seq_len, n_features]
        labels = labels.to(device)         # [batch]

        optimizer.zero_grad()

        aug_seq = augment(sequences)
        #seq2 = augment(sequences)
        #s = model(seq)
        #z2 = model(seq2)
        #features = torch.stack([z1, z2], dim=1)  # [B, 2, D]

        embeds = model(aug_seq)          # [batch, embed_dim]

        #print(embeds.shape)

        loss = criterion(embeds, labels) #criterion(embeds, labels)

        loss.backward()
        # Опционально: torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches


    if (epoch + 1) % print_every == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {avg_loss:.4f}  LR: {scheduler.get_last_lr()[0]:.2e}")

    # Можно сохранять чекпоинт``
    if (epoch + 1) % 3 == 0:
        torch.save(model.state_dict(), f"epochs\\gesture_embedder2_epoch_{epoch+1}.pth")

print("Обучение завершено.")

<>:69: SyntaxWarning: invalid escape sequence '\g'
<>:69: SyntaxWarning: invalid escape sequence '\g'
C:\Users\DARYA\AppData\Local\Temp\ipykernel_6992\4213289883.py:69: SyntaxWarning: invalid escape sequence '\g'
  torch.save(model.state_dict(), f"epochs\gesture_embedder2_epoch_{epoch+1}.pth")
  3%|▎         | 1/30 [38:25<18:34:05, 2305.03s/it]

Epoch [1/30]  Loss: 7.0288  LR: 7.63e-05


  7%|▋         | 2/30 [1:15:23<17:32:02, 2254.39s/it]

Epoch [2/30]  Loss: 6.9787  LR: 2.19e-04


 10%|█         | 3/30 [1:52:05<16:43:32, 2230.10s/it]

...

Epoch [28/30]  Loss: 5.4127  LR: 7.53e-06


 97%|█████████▋| 29/30 [17:55:44<38:55, 2335.37s/it]  

Epoch [29/30]  Loss: 5.4047  LR: 1.90e-06


 97%|█████████▋| 29/30 [18:07:18<37:29, 2249.59s/it]
C:\Users\DARYA\AppData\Local\Temp\ipykernel_6992\4213289883.py:69: SyntaxWarning: invalid escape sequence '\g'
  torch.save(model.state_dict(), f"epochs\gesture_embedder2_epoch_{epoch+1}.pth")


KeyboardInterrupt: 

In [71]:
torch.save(model.state_dict(), f"epochs\\gesture_embedder2_epoch.pth")

In [72]:
model1 = GestureModel()
checkpoint_path = "epochs\\gesture_embedder2_epoch.pth"
state_dict = torch.load(checkpoint_path, 
                        map_location=torch.device('cpu'),   # или 'cuda' если есть GPU
                        weights_only=True)

model1.load_state_dict(state_dict)
model1.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1.to(device)

GestureModel(
  (embedding): AllLandmarkEmbedding(
    (first_hand_emb): LandmarkEmbedding(
      (patch_embedding): PatchEmbedding(
        (proj): Linear(in_features=12, out_features=128, bias=True)
        (norm): RMSNorm()
      )
      (self_attention): MultiHeadAttention(
        (w_q): Linear(in_features=128, out_features=128, bias=False)
        (w_k): Linear(in_features=128, out_features=128, bias=False)
        (w_v): Linear(in_features=128, out_features=128, bias=False)
        (w_o): Linear(in_features=128, out_features=128, bias=False)
        (attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (adaptive_pool): AdaptiveAvgPool1d(output_size=1)
    )
    (sec_hand_emb): LandmarkEmbedding(
      (patch_embedding): PatchEmbedding(
        (proj): Linear(in_features=12, out_features=128, bias=True)
        (norm): RMSNorm()
 

 10%|█         | 1/10 [1:20:57<12:08:40, 4857.80s/it]
Epoch [1/10]  Loss: 13.7652  LR: 3.00e-04
Вторая эпоха шла более 120 минут и все еще не дошла.

In [44]:
import torch

def accuracy(output: torch.Tensor, target: torch.Tensor, topk=(1, 5)):
    """
    Вычисляет top-k accuracy.
    output: [batch_size, num_classes] — логиты или вероятности от модели
    target: [batch_size] — истинные метки (long)
    """
    with torch.no_grad():
        maxk = max(topk)                    # например 5
        batch_size = target.size(0)

        # Получаем индексы top-k предсказаний
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)  # [B, maxk]
        pred = pred.t()                    # [maxk, B]

        # Сравниваем с target
        correct = pred.eq(target.view(1, -1).expand_as(pred))  # [maxk, B]

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0)
            res.append(correct_k.mul_(100.0 / batch_size))   # в процентах

        return res   # возвращает список, например [top1, top5]

In [54]:
@torch.no_grad()
def validate(model, val_loader, device):
    model.eval()
    
    top1_correct = 0
    top5_correct = 0
    total = 0
    val_loss = 0.0
    
    criterion = nn.CrossEntropyLoss()   # для подсчёта loss (можно твой focal_loss)

    for sequences, labels in val_loader:
        sequences = sequences.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(sequences)                  # [B, num_classes]
        
        # Loss (опционально)
        loss = criterion(outputs, labels)
        val_loss += loss.item() * labels.size(0)
        
        # Accuracy
        acc1, acc5 = accuracy(outputs, labels, topk=(1, 5))
        
        top1_correct += acc1.item() * labels.size(0) / 100   # обратно в количество
        top5_correct += acc5.item() * labels.size(0) / 100
        total += labels.size(0)

    avg_val_loss = val_loss / total
    top1_acc = (top1_correct / total) * 100
    top5_acc = (top5_correct / total) * 100

    print(f"Validation Results:")
    print(f"   Loss: {avg_val_loss:.4f}")
    print(f"   Top-1 Accuracy: {top1_acc:.2f}%")
    print(f"   Top-5 Accuracy: {top5_acc:.2f}%")
    
    return top1_acc, top5_acc, avg_val_loss

In [67]:
print("=== Validation after 10 epochs ===")
val_top1, val_top5, val_loss = validate(model, val_loader, device)

=== Validation after 10 epochs ===
Validation Results:
   Loss: 5.4825
   Top-1 Accuracy: 4.05%
   Top-5 Accuracy: 9.63%


In [58]:
print("=== Validation after 10 epochs ===")
val_top1, val_top5, val_loss = validate(model, val_loader, device)

=== Validation after 10 epochs ===
Validation Results:
   Loss: 5.7949
   Top-1 Accuracy: 3.62%
   Top-5 Accuracy: 7.41%


In [73]:
print("=== Validation after 10 epochs ===")
val_top1, val_top5, val_loss = validate(model1, val_loader, device)

=== Validation after 10 epochs ===
Validation Results:
   Loss: 5.4825
   Top-1 Accuracy: 4.05%
   Top-5 Accuracy: 9.63%
